In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import accuracy_score

from sklearn.ensemble import GradientBoostingClassifier

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 1366)

#                                                       Theory
```js
        Read the complete theory of Gradient Boosting in Classification in "3. Gradient Boosting Classification.pdf" from Theory folder.
```

#                                   Dataset creation and Split into training and testing set.

In [2]:
# Hugeeeeeeee dataset.
X, y = make_classification(n_samples = 10_00000, n_features = 20, n_informative = 10, n_classes = 2, random_state = 42)

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

#                                           Gradient Boosting Classification from scratch.
```js
For fit():
1) For each tree model we need it’s previous model’s Predicted Output (probability), Residual Error (probability).

2)  And since for each model the prediction on output column (log(odds)) =

                                                             m (m = until the current model)
                base model’s predicted output (log(odds)) +  Σ (learning_rate * predicted_residual_error_i)
                                                            i=1

    we can store that calculation for each model, so we don’t have to do this from the first model(i=1) repeatedly. So for each model :
        
        predicted_y_for_current_model_log = predicted_y_for_current_model_log + (lr * predicted_residualError_for_current_model_log)
```

In [12]:
class GradientBoostingClassification:
    def __init__(self,  learning_rate: float = 0.1, n_estimators: int = 100, max_depth: int|None = 3,
                        max_leaf_nodes: int|None = None, random_state: int|None = None):
        # for gradient boosting.
        self.loss = "log_loss"
        self.learning_rate = learning_rate
        self.prediction_on_training = None # extra.

        # for tree, DecisionTreeRegressor.
        self.criterion = "squared_error" # by default for a DecisionTreeRegressor.
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.max_leaf_nodes = max_leaf_nodes
        self.random_state = random_state

        # for storing informations to predict().
        self.tree_models_leafNode_predResLog = np.array([dict()] * n_estimators)
        self.base_model_predicted_y_log = 0 # It is always a scaler value.
        self.tree_models = np.full(shape = (n_estimators), fill_value = None) # Array of Empty Trees.
    
    def fit(self, X: np.ndarray, y: np.ndarray):
        # for base model.
        base_model_pred_y_log = np.log((y == 1).sum() / (y == 0).sum())
        base_model_pred_y_prob = np.full(shape = y.shape, fill_value = 1 / (1 + np.exp(-base_model_pred_y_log)))
        base_model_resError_prob = y - base_model_pred_y_prob
        self.base_model_predicted_y_log = base_model_pred_y_log
        
        # Arrays of information we'll update for each model.
        prev_model_pred_y_prob = base_model_pred_y_prob
        prev_model_resError_prob = base_model_resError_prob
        pred_y_for_current_model_log = base_model_pred_y_log.copy() # Final Result for Base model.

        # for the rest tree models.
        for model_no in range(1, self.n_estimators + 1):
            tree = DecisionTreeRegressor(max_depth = self.max_depth, max_leaf_nodes = self.max_leaf_nodes,
                                         criterion = self.criterion, random_state = self.random_state)
            tree.fit(X, prev_model_resError_prob)
            self.tree_models[model_no - 1] = tree # collecting trees.

            leaf_nodes = tree.apply(X) # 1D numpy array.

            pred_res_for_current_model_log = np.full(shape = y.shape, fill_value = -1.0)
            for leaf in np.unique(leaf_nodes):
                mask = leaf_nodes == leaf
                pred_res_for_current_model_log[mask] = ( np.sum(prev_model_resError_prob[mask]) /
                                                         np.sum(prev_model_pred_y_prob[mask] * (1 - prev_model_pred_y_prob[mask]))
                                                       )
                
            # collecting current model's dict([leaf_nodes, corresponding_predicted_residual_error]).
            self.tree_models_leafNode_predResLog[model_no - 1] = dict(zip(leaf_nodes, pred_res_for_current_model_log))
                
            pred_y_for_current_model_log = pred_y_for_current_model_log + (self.learning_rate * pred_res_for_current_model_log)
            # Current Model's Y_Probability and Residual Error is Previous Model's Y_P.. and Res... Error for the next model.
            prev_model_pred_y_prob = 1 / (1 + np.exp(-pred_y_for_current_model_log)) # current_model_y_prob
            prev_model_resError_prob = y - prev_model_pred_y_prob
        
        # print(f"The Last Model's Output (probability) :\n{prev_model_pred_y_prob}.")

        self.prediction_on_training = np.where(prev_model_pred_y_prob < 0.5, 0, 1)
        return self
    
    def predict(self, X: np.ndarray):
        # all_models_predicted_residual_error_log = [[model1's res_log], [model2's res_log], [model3's res_log], ...].
        all_models_pred_res_log = np.full(shape = (X.shape[0], self.n_estimators), fill_value = 0.0)

        # Assigning each model's Residual Error (log(odds)) for all the rows of X.
        for m in range(self.n_estimators): # m = model no.
            leaf_nodes = self.tree_models[m].apply(X) # 1D Numpy array.
            corresponding_res_logs = np.array([self.tree_models_leafNode_predResLog[m][leaf_node] for leaf_node in leaf_nodes])
            all_models_pred_res_log[:, m] = corresponding_res_logs

        # calculating predicted_residual_error_log for each X's row and converting them into Probability in the next line.
        pred_y_log = self.base_model_predicted_y_log + np.sum(self.learning_rate * all_models_pred_res_log, axis = 1) # 1D Array.
        pred_y_prob = 1 / (1 + np.exp(-pred_y_log))

        return np.where(pred_y_prob < 0.5, 0, 1)

In [13]:
# Our custom GradientBoostingClassification. By default DecisionTreeRegressor uses criterion = 'squared_error'.
gbc = GradientBoostingClassification(max_depth = 8, n_estimators = 10, random_state = 42)
gbc.fit(x_train, y_train)

training_accuracy = accuracy_score(y_true = y_train, y_pred = gbc.predict(x_train))
testing_accuracy  = accuracy_score(y_true = y_test, y_pred  = gbc.predict(x_test))

print(f"Our custom GradientBoostingClassification's accuracy :")
print(f"For Training = {training_accuracy}, Testing = {testing_accuracy}.\n")

# Sklearn's GradientBoostingClassifier. Kept the parameters value same with our custom model.
gbc1 = GradientBoostingClassifier(criterion = 'squared_error', max_depth = 8, n_estimators = 10, random_state = 42)
gbc1.fit(x_train, y_train)

training_accuracy1 = accuracy_score(y_true = y_train, y_pred = gbc1.predict(x_train))
testing_accuracy1  = accuracy_score(y_true = y_test, y_pred  = gbc1.predict(x_test))

print(f"Sklearn's GradientBoostingClassifier's accuracy :")
print(f"For Training = {training_accuracy1}, Testing = {testing_accuracy1}.\n")

Our custom GradientBoostingClassification's accuracy :
For Training = 0.92805, Testing = 0.925665.

Sklearn's GradientBoostingClassifier's accuracy :
For Training = 0.92805, Testing = 0.925675.



#                                               fit() method with more details.
```js
            This is to show what is happening for each Tree Model inside GradientBoostingRegressor.
```

In [3]:
df1 = pd.DataFrame(data = {
        "iq": [10, 30, 20, 35, 40],
        "cg": [3.9, 3.2, 3.7, 2.7, 3.0],
        "placed": [0, 1, 1, 0, 1]
})

X, y = df1.iloc[:, :2], df1.iloc[:, 2]
df1

,iq,cg,placed
0,10,3.9,0
1,30,3.2,1
2,20,3.7,1
3,35,2.7,0
4,40,3.0,1


In [11]:
def fit_details(df2:pd.DataFrame, n_estimators:int = 2, lr: float = 0.1): # lr = learning_rate
    # for base models.
    df2['pred_placed_0(log)'] = np.log((df2['placed'] == 1).sum() / (df2['placed'] == 0).sum())
    df2['pred_placed_0(prob)'] = 1 / (1 + np.exp(-df2['pred_placed_0(log)']))
    df2['res_0(prob)'] = df2['placed'] - df2['pred_placed_0(prob)']
    
    # ans_df = [placed, pred_placed_0(prob), pred_placed_1(prob), pred_placed_1(prob) ...]
    ans_df = pd.DataFrame(data = {'placed':df2['placed'], 'pred_placed_0(prob)':df2['pred_placed_0(prob)']})
    pred_placed_for_current_model = df2['pred_placed_0(log)'].values # Final Result for Base model.

    for m in range(1, n_estimators + 1):
        tree1 = DecisionTreeRegressor(max_leaf_nodes = 3, random_state = 42) .fit(X, df2[f'res_{m-1}(prob)'])
        df2[f'leaf_node_{m}'] = tree1.apply(X)

        # Window function : groupby()[].transform(). work with one group and you'll understand what happens with other groups.
        df2[f'pred_res_{m}(log)'] = ( df2.groupby(by = f'leaf_node_{m}')[f'res_{m-1}(prob)'].transform('sum') / 
                df2.groupby(by = f'leaf_node_{m}')[f'pred_placed_{m-1}(prob)'].transform(lambda prev_prob: np.sum(prev_prob * (1 - prev_prob))) )
        
        pred_placed_for_current_model = pred_placed_for_current_model + (lr * df2[f'pred_res_{m}(log)'])
        df2[f'pred_placed_{m}(log)'] = pred_placed_for_current_model

        df2[f'pred_placed_{m}(prob)'] = 1 / (1 + np.exp(-df2[f'pred_placed_{m}(log)']))
        df2[f'res_{m}(prob)'] = df2['placed'] - df2[f'pred_placed_{m}(prob)']

        ans_df[f'pred_placed_{m}(prob)'] = df2[f'pred_placed_{m}(prob)']
    

    print(f"What actually happening for each model :\n\n{df2}.\n")
    print(f"{'-' * 250}")
    print(f"How predicted_placed(prob) is changing in each tree model :\n\n{ans_df}")

fit_details(df2 = df1)

What actually happening for each model :

   iq   cg  placed  pred_placed_0(log)  pred_placed_0(prob)  res_0(prob)  leaf_node_1  pred_res_1(log)  pred_placed_1(log)  pred_placed_1(prob)  res_1(prob)  leaf_node_2  pred_res_2(log)  pred_placed_2(log)  pred_placed_2(prob)  res_2(prob)
0  10  3.9       0            0.405465                  0.6         -0.6            1        -2.500000            0.155465             0.538788    -0.538788            1        -2.168201           -0.061355             0.484666    -0.484666
1  30  3.2       1            0.405465                  0.6          0.4            4         1.666667            0.572132             0.639255     0.360745            4         1.564321            0.728564             0.674490     0.325510
2  20  3.7       1            0.405465                  0.6          0.4            4         1.666667            0.572132             0.639255     0.360745            4         1.564321            0.728564             0.674490     0.3